# Setup

## Environment

In [1]:
import copy
import math
import random
import time
from collections import OrderedDict, defaultdict
from typing import Union, List

import numpy as np
import torch
from matplotlib import pyplot as plt
from torch import nn
from torch.optim import *
from torch.optim.lr_scheduler import *
from torch.utils.data import DataLoader
from torchprofile import profile_macs
from tqdm.auto import tqdm

import torch_pruning as tp

assert torch.cuda.is_available(), \
"The current runtime does not have CUDA support." \
"Please go to menu bar (Runtime - Change runtime type) and select GPU"

In [2]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

## Model & Dataset

In [3]:
model = torch.load("/home/Tommy/lab1/model/mobilenetv2_0.963.pth", map_location="cpu", weights_only=False)
model.cuda()

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [4]:
from torchvision.transforms import transforms
from torchvision.datasets import CIFAR10

transform = {
    "train": transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    "test": transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
}

dataset = {}
for split in ["train", "test"]:
  dataset[split] = CIFAR10(
    root="data/cifar10",
    train=(split == "train"),
    download=True,
    transform=transform[split],
  )

# You can apply your own batch_size
dataloader = {}
for split in ['train', 'test']:
  dataloader[split] = DataLoader(
    dataset[split],
    batch_size=64,
    shuffle=(split == 'train'),
    num_workers=0,
    pin_memory=True,
    drop_last=True
  )

Files already downloaded and verified
Files already downloaded and verified


In [5]:
example_inputs = torch.randn(1, 3, 224, 224).cuda()

## Methods

In [6]:
def train(
  model: nn.Module,
  dataloader: DataLoader,
  criterion: nn.Module,
  optimizer: Optimizer,
  scheduler: LambdaLR,
  # pruner = tp.pruner,
) -> None:
  model.train()
  # pruner.update_regularizer()
  for inputs, targets in tqdm(dataloader, desc='train', leave=False):
    # Move the data from CPU to GPU
    inputs = inputs.cuda()
    targets = targets.cuda()

    # Reset the gradients (from the last iteration)
    optimizer.zero_grad()

    # Forward inference
    outputs = model(inputs)
    loss = criterion(outputs, targets)

    # Backward propagation
    loss.backward()

    # Sparse training
    # pruner.regularize(model)

    # Update optimizer and LR scheduler
    optimizer.step()
    scheduler.step()

In [7]:
@torch.inference_mode()
def evaluate(
  model: nn.Module,
  dataloader: DataLoader,
  verbose=True,
) -> float:
  model.eval()

  num_samples = 0
  num_correct = 0

  for inputs, targets in tqdm(dataloader, desc="eval", leave=False,
                              disable=not verbose):
    # Move the data from CPU to GPU
    inputs = inputs.cuda()
    targets = targets.cuda()

    # Inference
    outputs = model(inputs)

    # Convert logits to class indices
    outputs = outputs.argmax(dim=1)

    # Update metrics
    num_samples += targets.size(0)
    num_correct += (outputs == targets).sum()

  return (num_correct / num_samples * 100).item()

In [8]:
def get_model_macs(model, inputs) -> int:
    return profile_macs(model, inputs)


def get_sparsity(tensor: torch.Tensor) -> float:
    """
    calculate the sparsity of the given tensor
        sparsity = #zeros / #elements = 1 - #nonzeros / #elements
    """
    return 1 - float(tensor.count_nonzero()) / tensor.numel()


def get_model_sparsity(model: nn.Module) -> float:
    """
    calculate the sparsity of the given model
        sparsity = #zeros / #elements = 1 - #nonzeros / #elements
    """
    num_nonzeros, num_elements = 0, 0
    for param in model.parameters():
        num_nonzeros += param.count_nonzero()
        num_elements += param.numel()
    return 1 - float(num_nonzeros) / num_elements

def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

In [9]:
def sparse_train(
  model: nn.Module,
  train_loader: DataLoader,
  val_loader: DataLoader,  # Add validation loader
  criterion: nn.Module,
  optimizer: Optimizer,
  scheduler: LambdaLR,
  pruner,
  epochs: int,
  save_path: str = "./model/best_sparse_training_model.pth"  # Path to save best model
) -> nn.Module:
    print("Sparse Training...")
    best_acc = 0.0

    for epoch in range(epochs):
        # Training phase
        model.train()
        pruner.update_regularizer()
        train_loss = 0.0

        for inputs, targets in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False):
            inputs, targets = inputs.cuda(), targets.cuda()
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            pruner.regularize(model)
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)
        scheduler.step()

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, targets in tqdm(val_loader, desc='Validation', leave=False):
                inputs, targets = inputs.cuda(), targets.cuda()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

        val_loss /= len(val_loader.dataset)
        acc = 100. * correct / total

        print(f'Epoch: {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Acc: {acc:.2f}%')

        # Save best model
        if acc > best_acc:
            best_acc = acc
            print(f'Saving best model with accuracy: {best_acc:.2f}%')
            torch.save(model.state_dict(), save_path)

    print(f'Best accuracy: {best_acc:.2f}%')
    return model

# Prunning by Torch_prunning

In [10]:
pruned_model = copy.deepcopy(model)

In [11]:
# 0. importance criterion for parameter selections
# imp = tp.importance.MagnitudeImportance()
imp = tp.importance.TaylorImportance()

In [12]:
# 1. ignore some layers that should not be pruned, e.g., the final classifier layer.
ignored_layers = []
for m in pruned_model.modules():
    if isinstance(m, torch.nn.Linear) and m.out_features == 10:
        ignored_layers.append(m) # DO NOT prune the final classifier!

ignored_layers.append(pruned_model.features[1].conv[0][0])
ignored_layers.append(pruned_model.features[2].conv[1][0])
ignored_layers.append(pruned_model.features[2].conv[2])
ignored_layers.append(pruned_model.features[4].conv[2])
ignored_layers.append(pruned_model.features[5].conv[1][0])
ignored_layers.append(pruned_model.features[6].conv[1][0])

print(f'ignored layers: {ignored_layers}')

ignored layers: [Linear(in_features=1280, out_features=10, bias=True), Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False), Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=96, bias=False), Conv2d(96, 24, kernel_size=(1, 1), stride=(1, 1), bias=False), Conv2d(144, 32, kernel_size=(1, 1), stride=(1, 1), bias=False), Conv2d(192, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=192, bias=False), Conv2d(192, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=192, bias=False)]


In [13]:
pruning_ratio_dict = {
  pruned_model.features[7].conv[2]: 0.90,
  pruned_model.features[8].conv[0][0]: 0.90,
  pruned_model.features[8].conv[2]: 0.90,
  pruned_model.features[9].conv[0][0]: 0.90,
  pruned_model.features[9].conv[2]: 0.90,
  pruned_model.features[10].conv[0][0]: 0.90,
  pruned_model.features[10].conv[2]: 0.90,
  pruned_model.features[11].conv[0][0]: 0.90,
  pruned_model.features[11].conv[2]: 0.050,
  pruned_model.features[12].conv[0][0]: 0.90,
  pruned_model.features[12].conv[2]: 0.90,
  pruned_model.features[13].conv[0][0]: 0.90,
  pruned_model.features[13].conv[2]: 0.080,
  pruned_model.features[14].conv[0][0]: 0.075,
  # pruned_model.features[14].conv[2]: 0.90,
  pruned_model.features[15].conv[0][0]: 0.90,
  pruned_model.features[15].conv[2]: 0.90,
  pruned_model.features[16].conv[0][0]: 0.90,
  pruned_model.features[16].conv[2]: 0.085,
  pruned_model.features[17].conv[0][0]: 0.95,
  pruned_model.features[17].conv[2]: 0.90,
  pruned_model.features[18]: 0.98,
}

In [14]:
# 2. Pruner initialization
iterative_steps = 100
pruner = tp.pruner.GroupNormPruner(
    pruned_model, 
    example_inputs, 
    isomorphic=True,
    global_pruning=True,
    importance=imp,
    iterative_steps=iterative_steps,
    pruning_ratio=0.1,
    pruning_ratio_dict=pruning_ratio_dict,
    ignored_layers=ignored_layers,
    round_to=4,
)

In [ ]:
sparset_train_epochs = 50
optimizer = torch.optim.SGD(pruned_model.parameters(), lr=0.0001, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, sparset_train_epochs)
criterion = nn.CrossEntropyLoss()

if isinstance(pruner, tp.pruner.GroupNormPruner):
  sparse_train(pruned_model, dataloader['train'], dataloader['test'], criterion, optimizer, scheduler, pruner, sparset_train_epochs)
  pruned_model.load_state_dict(torch.load("./model/best_sparse_training_model.pth"))

Sparse Training...


Epoch 1/50:   0%|          | 0/781 [00:00<?, ?it/s]

In [ ]:
num_finetune_epochs = 20
optimizer = torch.optim.SGD(pruned_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_finetune_epochs)

base_macs, base_nparams = tp.utils.count_ops_and_params(model, example_inputs)
for i in range(iterative_steps):

    # Compute gradients for TaylorImportance
    if isinstance(imp, tp.importance.TaylorImportance):
        inputs, labels = next(iter(dataloader['train']))
        outputs = pruned_model(inputs.cuda())
        loss = criterion(outputs, labels.cuda())
        loss.backward()  # Critical for TaylorImportance

    # 3. the pruner.step will remove some channels from the model with least importance
    pruner.step()
    
    # 4. Do whatever you like here, such as fintuning
    macs, nparams = tp.utils.count_ops_and_params(pruned_model, example_inputs)
    print(pruned_model(example_inputs).shape)
    print(
        "  Iter %d/%d, Params: %.2f M => %.2f M"
        % (i+1, iterative_steps, base_nparams / 1e6, nparams / 1e6)
    )
    print(
        "  Iter %d/%d, MACs: %.2f M => %.2f M"
        % (i+1, iterative_steps, base_macs / 1e6, macs / 1e6)
    )
    print(f'Finetuning Pruned Sparse Model')
    best_accuracy = 0
    best_sparse_model = None
    for epoch in range(num_finetune_epochs):
        # At the end of each train iteration, we have to apply the pruning mask
        #    to keep the model sparse during the training
        train(pruned_model, dataloader['train'], criterion, optimizer, scheduler) 
        accuracy = evaluate(pruned_model, dataloader['test'])

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_sparse_model = copy.deepcopy(pruned_model.state_dict())
        print(f'    Epoch {epoch+1} Accuracy {accuracy:.2f}% / Best Accuracy: {best_accuracy:.2f}%')
    
    if best_sparse_model is not None:
        pruned_model.load_state_dict(best_sparse_model)


torch.Size([1, 10])
  Iter 1/100, Params: 2.24 M => 2.16 M
  Iter 1/100, MACs: 318.97 M => 303.23 M
Finetuning Pruned Sparse Model


train:   0%|          | 0/781 [00:00<?, ?it/s]

eval:   0%|          | 0/156 [00:00<?, ?it/s]

    Epoch 1 Accuracy 94.58% / Best Accuracy: 94.58%


train:   0%|          | 0/781 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Evaluation Criteria

In [ ]:
ops, size = tp.utils.count_ops_and_params(pruned_model, example_inputs=example_inputs)
MFLOPs = ops/1e6
print("MELOPs: {:0.2f}".format(MFLOPs))
Accuracy = evaluate(pruned_model, dataloader['test'], verbose=False)
print("Accuracy: {:0.2f}%".format(Accuracy))
Accuracy = Accuracy/100

score = max(0, (200-MFLOPs)/(200-45) - 10*min(max(0.92-Accuracy, 0), 1)) * 45
print("Your Score: {:0.2f}/45".format(score))

In [ ]:
torch.save(pruned_model, "./model/mobilenet_M{:.2f}_A{:.2f}.pth".format(MFLOPs, Accuracy*100))